In [1]:
###Load packages###
import pandas as pd
import os
import ast
from scipy import stats
from matplotlib import pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
import numpy as np
import statsmodels.formula.api as smf
import seaborn as sns


###Load cleaned dataset###

#Set file paths
topdir = '/Users/sm6511/Desktop/Prediction-Accomodation-Exp'
study = 'Study3.0'
cleandir = os.path.join(topdir, f'data/{study}/Cleaned')
outputdir = os.path.join(topdir, f'Analysis/{study}')
outputdirCombined = os.path.join(topdir, f'data/Combined')
outputdirCleaned = os.path.join(topdir, f'data/{study}/Cleaned')
os.makedirs(outputdir, exist_ok=True)

#Read in cleaned data 
accomodate_path = os.path.join(cleandir, f'{study}Accommodate.csv')
predict_path   = os.path.join(cleandir, f'{study}Predict.csv')

df_accommodate = pd.read_csv(accomodate_path)
df_predict   = pd.read_csv(predict_path)

df_accommodate['task'] = 'accommodate'
df_predict['task']   = 'predict'


print("Accommodate columns:", df_accommodate.columns.tolist())
print("Predict columns:", df_predict.columns.tolist())


Accommodate columns: ['participant', 'free_texts', 'feedback', 'fertility_score', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_feet', 'training_color', 'testing_categories', 'conditionOrder', 'training_image_order', 'attention_check', 'relevant_dim', 'irrelevant_dim', 'feet_high', 'color_low', 'color_high', 'feet_low', 'feet_discrete_slider.response', 'feet_direction_slider.response', 'feet_continuous_slider.response', 'color_discrete_slider.response', 'color_direction_slider.response', 'color_continuous_slider.response', 'task']
Predict columns: ['participant', 'training_responses', 'fertility_score', 'error', 'feedback', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_feet', 'training_color', 'testing_categories', 'conditionOrder', 'training_image_order', 'attention_check', 'relevant_dim', 'irrelevant_dim', 'feet_high', 'color_low', 'color_high', 'feet_low', 'feet_discrete_slider.respon

In [2]:
#Converting string representations of lists back to lists

def parse_list_column(x):
    """take column entries that are strings representing lists and convert them to actual lists"""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            return ast.literal_eval(x)
        else:
            return [x]
    return []
for col in ['training_feet', 'training_color', 'training_image_order', 'training_categories', 'testing_categories']:
    df_accommodate[col] = df_accommodate[col].apply(parse_list_column)
    df_predict[col]   = df_predict[col].apply(parse_list_column)

df_accommodate['testing_responses'] = df_accommodate['testing_responses'].apply(ast.literal_eval)
df_accommodate['fertility_score'] = df_accommodate['fertility_score'].apply(ast.literal_eval)
df_accommodate['testing_image_order'] = df_accommodate['testing_image_order'].apply(ast.literal_eval)
df_predict['testing_responses'] = df_predict['testing_responses'].apply(ast.literal_eval)
df_predict['fertility_score'] = df_predict['fertility_score'].apply(ast.literal_eval)
df_predict['testing_image_order'] = df_predict['testing_image_order'].apply(ast.literal_eval)
#Combine the dataframes and create an arbitrary column for participant numbering (the yoked orders are already stored in 'conditionOrder')
df_combined = pd.concat([df_accommodate, df_predict], ignore_index=True)
df_combined['participant'] = range(1, len(df_combined) + 1)



In [6]:
import pandas as pd
from scipy.stats import chi2_contingency

participant_rows = []

for _, row in df_combined.iterrows():
    feet_yes = 1 if str(row['feet_discrete_slider.response']).strip() == 'Yes' else 0
    color_yes = 1 if str(row['color_discrete_slider.response']).strip() == 'Yes' else 0

    model_param_score = feet_yes + color_yes

    participant_rows.append({
        'participant': row['participant'],
        'task': row['task'],
        'model_param_score': model_param_score,
        'conditionOrder': row['conditionOrder'],

        'relevant_dim': row['relevant_dim'],
        'irrelevant_dim': row['irrelevant_dim'],

        'feet_high': row['feet_high'],
        'color_high': row['color_high'],
        'feet_low': row['feet_low'],
        'color_low': row['color_low'],

        'feet_discrete_slider.response': row['feet_discrete_slider.response'],
        'color_discrete_slider.response': row['color_discrete_slider.response'],

        'feet_reported_relevant': feet_yes,
        'color_reported_relevant': color_yes,

        # overfit = selected both dimensions
        'overfit': model_param_score == 2
    })

df_params = pd.DataFrame(participant_rows)
df_params.to_csv(os.path.join(outputdirCombined, f'df_params_study3_for_r.csv'), index=False)

contingency = pd.crosstab(
    df_params['task'],
    df_params['overfit']
)

print(contingency)

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")

overfit      False  True 
task                     
accommodate    122     95
predict        150     67
Chi-square = 7.180
df = 1
p-value = 0.0074


In [4]:
contingency_all = pd.crosstab(
    df_params['task'],
    df_params['model_param_score']
)

print(contingency_all)

chi2, p, dof, expected = chi2_contingency(contingency_all)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")

expected_df = pd.DataFrame(
    expected,
    index=contingency_all.index,
    columns=contingency_all.columns
)

# Row and column proportions
n = contingency_all.values.sum()
row_prop = contingency_all.sum(axis=1) / n
col_prop = contingency_all.sum(axis=0) / n

# Adjusted standardized residuals
adj_resid = (
    contingency_all - expected_df
) / np.sqrt(
    expected_df *
    (1 - row_prop.values[:, None]) *
    (1 - col_prop.values[None, :])
)

print("Observed:")
print(contingency_all)

print("\nExpected:")
print(expected_df.round(2))

print("\nAdjusted standardized residuals:")
print(adj_resid.round(2))
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# p values
pvals = 2 * norm.sf(np.abs(adj_resid.values))

# Holm correction 
_, pvals_holm, _, _ = multipletests(
    pvals.flatten(),
    method="holm"
)

pvals_holm = pd.DataFrame(
    pvals_holm.reshape(adj_resid.shape),
    index=adj_resid.index,
    columns=adj_resid.columns
)

print("\nHolm-adjusted p-values:")
print(pvals_holm.round(4))

model_param_score   0    1   2
task                          
accommodate        23   99  95
predict            41  109  67
Chi-square = 10.383
df = 2
p-value = 0.0056
Observed:
model_param_score   0    1   2
task                          
accommodate        23   99  95
predict            41  109  67

Expected:
model_param_score     0      1     2
task                                
accommodate        32.0  104.0  81.0
predict            32.0  104.0  81.0

Adjusted standardized residuals:
model_param_score     0     1     2
task                               
accommodate       -2.44 -0.96  2.78
predict            2.44  0.96 -2.78

Holm-adjusted p-values:
model_param_score       0       1       2
task                                     
accommodate        0.0593  0.6732  0.0327
predict            0.0593  0.6732  0.0327


Importance scores analyses next

In [5]:

feet_map = {
    'round': 'f',
    'curly/pointy': 'c'
}

color_map = {
    'red': 'y',
    'green': 'g'
}



feature_maps = {
    'feet': feet_map,
    'color': color_map
}

In [ ]:
#Compute feature importance scores

from doctest import debug


def compute_feature_importance_from_df(df):
    """
    Compute numeric feature importance scores(-7 to 7) for each participant,
    based on the saved slider_responses and the feature _high/_low mapping.
    This is computed based on whether a feature was really relevant (positive sign) or irrelevant (negative sign).
    0 = no response or feature was not thought to be relevant
    """
    features = ['feet', 'color']
    
    def compute_row_importance(row, feat):
        disc = row[f'{feat}_discrete_slider.response']
        dirc = row[f'{feat}_direction_slider.response']
        cont = row[f'{feat}_continuous_slider.response']

        #If they said a feature wasn't relevant, then importance is 0
        
        if disc == 'No' or pd.isna(disc):
            return 0.0
        
        # Make sure continuous slider value exists, if not, set it to 0
        cont_val = float(cont) if not pd.isna(cont) else 0.0

        # Get mapping from long to short feature name
        mapping = feature_maps.get(feat, {})

        # Normalize strings: strip whitespace, collapse multiple spaces, lower-case
        def normalize_str(s):
            """Strip leading/trailing whitespace and collapse internal multiple spaces."""

            if isinstance(s, str):
                return " ".join(s.split()).lower()
            return ""
        

        #Name of features need to be normalized for comparison using the mapping
        dirc_norm = normalize_str(dirc)

        #Get internal short code for selected feature direction
        internal_dirc = mapping.get(dirc_norm, None)
        

        high_val = normalize_str(row[f'{feat}_high'])
        low_val  = normalize_str(row[f'{feat}_low'])
        

        
        # Debug print statement (make sure mappings look right)
        debug = True
        if debug:
            print('response:', repr(dirc_norm), 'internal:', repr(internal_dirc), 
                'high:', repr(high_val), 'low:', repr(low_val))
            

        #If they correctly selected the high feature, assign positive sign
        if internal_dirc == high_val:
            sign = 1
        #If they incorrectly selected the low feature, assign negative sign
        elif internal_dirc == low_val:
            if debug:
                print('in negative')
            sign = -1
        else:
            if debug:
                print('in empty')
            sign = 0
            cont_val = 0.0

        # Add sign to continuous value
        importance = cont_val * sign

        return importance

    
    # Compute for each feature
    for feat in features:
        df[f'{feat}_importance'] = df.apply(lambda row: compute_row_importance(row, feat), axis=1)
    
    return df

df_combined = compute_feature_importance_from_df(df_combined)
#df_filtered = compute_feature_importance_from_df(df_filtered)
if debug:
    print(df_combined['feet_importance'])

response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'curly/pointy' internal: 'c' high: 'c' low: 'f'
response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'curly/pointy' internal: 'c' high: 'f' low: 'c'
in negative
response: 'curly/pointy' internal: 'c' high: 'f' low: 'c'
in negative
response: 'curly/pointy' internal: 'c' high: 'f' low: 'c'
in negative
response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'curly/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curly/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curly/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curly/pointy' internal: 'c' high: 'f' low: 'c'
in negative
response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'round' internal: 'f' high: 'f' low: 'c'
response: 'curly/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curly/pointy' internal: 'c' high: 'c' low: 'f'
response: 'curl

In [7]:
import pandas as pd
"""Reshape to long format with 1 row per participant x feature dimension"""
# Keep only necessary columns
cols_to_keep = [
    'participant', 'task', 
    'feet_importance', 'color_importance',
    'relevant_dim', 'irrelevant_dim', 'feet_high','color_high', 'feet_discrete_slider.response'
]

df_long = df_combined[cols_to_keep].copy()

# Melt importance columns
df_long = df_long.melt(
    id_vars=['participant', 'task', 'relevant_dim', 'irrelevant_dim', 'color_high', 'feet_high', 'feet_discrete_slider.response'],
    value_vars=['feet_importance', 'color_importance'],
    var_name='feature_dimension',
    value_name='feature_importance'
)

# Simplify feature dimension names
df_long['feature_dimension'] = df_long['feature_dimension'].str.replace('_importance','')

def get_relevance(row):
    if row['feature_dimension'] in [row['relevant_dim']]:
        return 'relevant'
    else:
        return 'irrelevant'

df_long['feature_relevance'] = df_long.apply(get_relevance, axis=1)

print(df_long.tail(20))

     participant     task relevant_dim irrelevant_dim color_high feet_high  \
848          415  predict        color           feet          Y         F   
849          416  predict         feet          color          Y         C   
850          417  predict         feet          color          Y         C   
851          418  predict         feet          color          G         F   
852          419  predict        color           feet          G         F   
853          420  predict         feet          color          Y         C   
854          421  predict        color           feet          Y         F   
855          422  predict         feet          color          Y         F   
856          423  predict        color           feet          G         C   
857          424  predict         feet          color          Y         C   
858          425  predict        color           feet          G         C   
859          426  predict         feet          color          Y

In [ ]:
print(df_long)
df_long.to_csv(os.path.join(outputdirCombined, 'df_long_for_R-Study3.csv'), index=False)

Next, get the correlations/errors

In [8]:
#Group by average food amount per item in training
df = df_combined[['task', 'training_image_order', 'fertility_score', 'conditionOrder']]
df_long2 = (
    df
    .explode(['training_image_order', 'fertility_score'])
    .rename(columns={'training_image_order': 'item'})
)
avg_food = (
    df_long2
    .groupby(['task', 'conditionOrder', 'item'], as_index=False)
    ['fertility_score']
    .mean()
)
avg_food_train = avg_food.copy()
print(avg_food_train.head(20))


           task  conditionOrder item fertility_score
0   accommodate               1  G_C            6.75
1   accommodate               1  G_F             2.5
2   accommodate               1  Y_C            7.75
3   accommodate               1  Y_F             2.5
4   accommodate               2  G_C            6.75
5   accommodate               2  G_F             7.5
6   accommodate               2  Y_C             3.0
7   accommodate               2  Y_F             3.5
8   accommodate               3  G_C            2.25
9   accommodate               3  G_F             8.0
10  accommodate               3  Y_C             3.5
11  accommodate               3  Y_F            5.75
12  accommodate               4  G_C            3.75
13  accommodate               4  G_F             5.5
14  accommodate               4  Y_C             7.0
15  accommodate               4  Y_F            7.75
16  accommodate               5  G_C            8.25
17  accommodate               5  G_F          

In [9]:
#Get fertility ratings by item

df = df_combined[['task', 'conditionOrder', 'testing_image_order', 'testing_responses',
                  'relevant_dim', 'irrelevant_dim', 'color_high', 'feet_high']]
df_long2 = (
    df
    .explode(['testing_image_order', 'testing_responses'])
    .rename(columns={'testing_image_order': 'item'})
)
avg_food_test = df_long2.copy()
print(avg_food_test)


            task  conditionOrder item testing_responses relevant_dim  \
0    accommodate             183  Y_C               8.0         feet   
0    accommodate             183  G_F               8.0         feet   
0    accommodate             183  Y_F               5.0         feet   
0    accommodate             183  G_C               3.0         feet   
1    accommodate              75  Y_C               5.0         feet   
..           ...             ...  ...               ...          ...   
432      predict             204  G_F               2.0        color   
433      predict             120  G_F               4.0        color   
433      predict             120  G_C               2.0        color   
433      predict             120  Y_C               7.0        color   
433      predict             120  Y_F               8.0        color   

    irrelevant_dim color_high feet_high  
0            color          Y         F  
0            color          Y         F  
0        

In [10]:
#Now merge the two (actual food amounts in training vs ratings in testing) and compute error
df_merged = avg_food_test.merge(
    avg_food_train,
    on=['task', 'conditionOrder', 'item'],
    how='left'
)

#Add Error and absolute error
df_merged['error'] = (
    df_merged['testing_responses'] - df_merged['fertility_score']
)
df_merged['abs_error'] = df_merged['error'].abs()
df_merged[['color', 'feet']] = df_merged['item'].str.split('_', expand=True)

print(df_merged)

             task  conditionOrder item testing_responses relevant_dim  \
0     accommodate             183  Y_C               8.0         feet   
1     accommodate             183  G_F               8.0         feet   
2     accommodate             183  Y_F               5.0         feet   
3     accommodate             183  G_C               3.0         feet   
4     accommodate              75  Y_C               5.0         feet   
...           ...             ...  ...               ...          ...   
1731      predict             204  G_F               2.0        color   
1732      predict             120  G_F               4.0        color   
1733      predict             120  G_C               2.0        color   
1734      predict             120  Y_C               7.0        color   
1735      predict             120  Y_F               8.0        color   

     irrelevant_dim color_high feet_high fertility_score error abs_error  \
0             color          Y         F       

In [11]:
df_merged.to_csv(os.path.join(outputdirCombined, 'df_merged_for_R_Study3.csv'), index=False)

In [12]:
# Columns indicating whether the item's feature is the "high" dimension (1 or 0 coding)
df_merged['color_match_high']  = (df_merged['color']  == df_merged['color_high']).astype(int)
df_merged['feet_match_high'] = (df_merged['feet'] == df_merged['feet_high']).astype(int)
#print(df_merged.head(20))
# Group by participant
participant_corrs = []

for pid, g in df_merged.groupby(['task', 'conditionOrder']):
    for feat in ['color','feet']:
        # Column indicating match to high value
        match_col = f"{feat}_match_high"
        
        # Compute correlation
        corr = g['testing_responses'].corr(g[match_col])
        
        # Determine if this feature is relevant for this participant
        relevant = g['relevant_dim'].iloc[0] == feat
        high_col  = f"{feat}_high"
        
        # Store
        participant_corrs.append({
            'participant': pid,
            'task': g['task'].iloc[0],
            'feature_dimension': feat,
            'high_value': g[high_col].iloc[0],
            'feature_relevance': 'relevant' if relevant else 'irrelevant',
            'correlation': corr,
            'irrelevant_dim': g['irrelevant_dim'].iloc[0],
            'abs_correlation': abs(corr) if pd.notna(corr) else None
        })

df_corr = pd.DataFrame(participant_corrs)
print(df_corr.tail(40))



        participant     task feature_dimension high_value feature_relevance  \
828  (predict, 201)  predict             color          Y        irrelevant   
829  (predict, 201)  predict              feet          C          relevant   
830  (predict, 202)  predict             color          G          relevant   
831  (predict, 202)  predict              feet          C        irrelevant   
832  (predict, 203)  predict             color          G        irrelevant   
833  (predict, 203)  predict              feet          F          relevant   
834  (predict, 204)  predict             color          Y          relevant   
835  (predict, 204)  predict              feet          C        irrelevant   
836  (predict, 205)  predict             color          G        irrelevant   
837  (predict, 205)  predict              feet          C          relevant   
838  (predict, 206)  predict             color          G          relevant   
839  (predict, 206)  predict              feet      

In [14]:
# Get the actual correlations between fertility score and feature match
# for each participant in training

actual_corrs = []

for (task, condition_order), g in df_merged.groupby(
    ["task", "conditionOrder"]
):

    participant = (task, condition_order)
    relevant_dim = g["relevant_dim"].iloc[0]

    for feat in ["color", "feet"]:

        match_col = f"{feat}_match_high"

        corr = g["fertility_score"].corr(g[match_col])

        actual_corrs.append({
            "participant": participant,
            "task": task,
            "conditionOrder": condition_order,
            "feature_dimension": feat,
            "feature_relevance": (
                "relevant"
                if feat == relevant_dim
                else "irrelevant"
            ),
            "actual_correlation": corr
        })

actual_corrs = pd.DataFrame(actual_corrs)

df_corr = df_corr.merge(
    actual_corrs[
        [
            "participant",
            "feature_dimension",
            "actual_correlation"
        ]
    ],
    on=["participant", "feature_dimension"],
    how="left"
)

print(df_corr.head(20))

          participant         task feature_dimension high_value  \
0    (accommodate, 1)  accommodate             color          Y   
1    (accommodate, 1)  accommodate              feet          C   
2    (accommodate, 2)  accommodate             color          G   
3    (accommodate, 2)  accommodate              feet          F   
4    (accommodate, 3)  accommodate             color          G   
5    (accommodate, 3)  accommodate              feet          F   
6    (accommodate, 4)  accommodate             color          Y   
7    (accommodate, 4)  accommodate              feet          F   
8    (accommodate, 5)  accommodate             color          Y   
9    (accommodate, 5)  accommodate              feet          C   
10   (accommodate, 6)  accommodate             color          G   
11   (accommodate, 6)  accommodate              feet          F   
12   (accommodate, 7)  accommodate             color          G   
13   (accommodate, 7)  accommodate              feet          

In [15]:
df_corr.to_csv(os.path.join(outputdirCombined, 'df_corr_for_R_Study3.csv'), index=False)